In [2]:
import json
import pandas as pd
import numpy as np
import altair as alt
from scipy import stats

### Setup (Loading data, minor preprocesing)

In [24]:
with open('dataset.json', 'r') as f:
    raw = json.load(f)

RESOLUTIONS = ['50', '71', '87', '100']
PARAMS = ['alpha_weight', 'hist_percent', 'filter_size', 'num_samples']

records = []

for scene, scene_data in raw.items():
    if not isinstance(scene_data, dict):
        continue
    for res in RESOLUTIONS:
        if res not in scene_data:
            continue
        for ref, ref_data in scene_data[res].items():
            if ref.startswith('ref-'):
                for param in PARAMS:
                    if param not in ref_data:
                        continue
                    for entry in ref_data[param]:
                        records.append({
                            'scene': scene,
                            'resolution': int(res),
                            'parameter': param,
                            'value': entry['value'],
                            'score': entry['score'],
                        })

df = pd.DataFrame(records)

# Mean-center scores within each (scene, parameter, resolution) group
# Done so that differences in baseline scores for scenes don't confound comparisons
df['scene_param_mean'] = df.groupby(['scene', 'parameter', 'resolution'])['score'].transform('mean')
df['score_centered'] = df['score'] - df['scene_param_mean']

# Ignore hist percent less than 100 (measurement exists from prev trials but is not significant)
df = df[~((df['parameter'] == 'hist_percent') & (df['value'] < 100))]

# These are the weird outlier scenes--we cut them
EXCLUDE_SCENES = ['junkyard-mound1', 'junkyard-mound2', 'oldmine-speed-18', 'oldmine-speed-35', 'oldmine-speed-75', 'oldmine-speed-9', 'oldmine-warm']
df = df[~df['scene'].isin(EXCLUDE_SCENES)]

# Print for sanity check
print(df.shape)
df.head(10)

(2376, 7)


,scene,resolution,parameter,value,score,scene_param_mean,score_centered
0,abandoned,50,alpha_weight,0.01,91.475502,91.606316,-0.130814
1,abandoned,50,alpha_weight,0.02,91.696266,91.606316,0.089950
2,abandoned,50,alpha_weight,0.04,92.162430,91.606316,0.556114
3,abandoned,50,alpha_weight,0.06,92.602554,91.606316,0.996238
4,abandoned,50,alpha_weight,0.10,93.051781,91.606316,1.445464
5,abandoned,50,alpha_weight,0.20,93.491249,91.606316,1.884933
6,abandoned,50,alpha_weight,0.50,92.588005,91.606316,0.981689
7,abandoned,50,alpha_weight,0.60,91.999229,91.606316,0.392913
8,abandoned,50,alpha_weight,0.70,91.279709,91.606316,-0.326607
9,abandoned,50,alpha_weight,0.80,90.529510,91.606316,-1.076806


In [25]:
df.groupby(['parameter', 'resolution'])['score'].agg(['count', 'mean', 'std']).round(3)

count    mean    std
parameter    resolution                      
alpha_weight 50            264  86.114  8.864
             71            252  91.326  5.985
             87            252  93.199  4.957
             100           252  94.277  4.443
filter_size  50            153  87.710  7.409
             71            147  91.156  6.175
             87            147  92.444  5.636
             100           146  93.339  5.279
hist_percent 50             88  87.632  7.304
             71             84  91.853  5.674
             87             84  93.219  5.144
             100            82  94.015  4.814
num_samples  50            110  87.554  7.599
             71            105  91.172  6.186
             87            105  92.459  5.639
             100           105  93.279  5.299

### Basic Info about Data

In [26]:
print(f"Scenes: {df['scene'].nunique()}")
print(f"Scene names: {df['scene'].unique().tolist()}")
print(f"\nParameter value counts:")
print(df.groupby(['parameter', 'value'])['scene'].count().rename('n_scenes'))

Scenes: 21
Scene names: ['abandoned', 'abandoned-demo', 'abandoned-flipped', 'cubetest', 'fantasticvillage-open', 'lightfoliage', 'lightfoliage-close', 'oldmine', 'quarry-all', 'quarry-rocksonly', 'resto-close', 'resto-fwd', 'resto-pan', 'scifi', 'subway-lookdown', 'subway-turn', 'wildwest-bar', 'wildwest-barzoom', 'wildwest-behindcounter', 'wildwest-store', 'wildwest-town']

Parameter value counts:
parameter     value 
alpha_weight  0.01      85
              0.02      85
              0.04      85
              0.06      85
              0.10      85
              0.20      85
              0.50      85
              0.60      85
              0.70      85
              0.80      85
              0.90      85
              1.00      85
filter_size   0.10      85
              0.25      85
              0.50      84
              0.70      84
              0.90      85
              0.95      85
              1.00      85
hist_percent  100.00    85
              125.00    85
         

In [ ]:
# Print out for each scene x resolution, whether the param is calculated 
for param in PARAMS:
    sub = df[df['parameter'] == param]
    pivot = sub.groupby(['scene', 'resolution', 'value'])['score'].count().unstack('value')
    print(f"\n=== {param} ===")
    print(pivot.to_string())

In [32]:
# Find missing params
for param in PARAMS:
    sub = df[df['parameter'] == param]
    pivot = sub.groupby(['scene', 'resolution', 'value'])['score'].count().unstack('value')
    missing = pivot[pivot.isna().any(axis=1)]
    if not missing.empty:
        print(f"\n=== {param} ===")
        print(missing.to_string())


=== hist_percent ===
value                      100.0  125.0  150.0  200.0
scene          resolution                            
abandoned-demo 100           1.0    1.0    NaN    1.0
cubetest       100           1.0    1.0    1.0    NaN

=== filter_size ===
value                          0.10  0.25  0.50  0.70  0.90  0.95  1.00
scene              resolution                                          
abandoned-flipped  100          1.0   1.0   NaN   1.0   1.0   1.0   1.0
lightfoliage-close 50           1.0   1.0   1.0   NaN   1.0   1.0   1.0


In [30]:
dupe = df[(df['scene'] == 'fantasticvillage-open')]
print(dupe.groupby(['parameter', 'resolution', 'value'])['score'].count())

parameter     resolution  value
alpha_weight  50          0.01     2
                          0.02     2
                          0.04     2
                          0.06     2
                          0.10     2
                                  ..
num_samples   100         4.00     1
                          8.00     1
                          16.00    1
                          32.00    1
                          64.00    1
Name: score, Length: 112, dtype: int64


### Statistical Significance of Parameters

In [14]:
anova_records = []

for (param, res), group in df.groupby(['parameter', 'resolution']):
    groups_by_value = [g['score'].values for _, g in group.groupby('value')]
    if len(groups_by_value) < 2:
        continue
    f_stat, p_val = stats.f_oneway(*groups_by_value)
    anova_records.append({
        'parameter': param,
        'resolution': res,
        'F': round(f_stat, 4),
        'p_value': round(p_val, 6),
        'significant': p_val < 0.05,
    })

anova_df = pd.DataFrame(anova_records)
anova_df

,parameter,resolution,F,p_value,significant
0,alpha_weight,50,1.2680,0.242426,False
1,alpha_weight,71,0.4111,0.950656,False
2,alpha_weight,87,0.4394,0.937330,False
3,alpha_weight,100,1.7656,0.058943,False
4,filter_size,50,0.0332,0.999842,False
5,filter_size,71,0.0086,0.999997,False
6,filter_size,87,0.0036,1.000000,False
7,filter_size,100,0.1740,0.983547,False
8,hist_percent,50,0.0255,0.994475,False
9,hist_percent,71,0.0883,0.966288,False


In [15]:
# Aggregate across scenes (mean score per param × value × resolution)
agg = df.groupby(['parameter', 'resolution', 'value'])['score'].agg(['mean', 'std', 'count']).reset_index()
agg['se'] = agg['std'] / np.sqrt(agg['count'])  # standard error
agg['ci95'] = 1.96 * agg['se']
agg['resolution'] = agg['resolution'].astype(str) + '%'

charts = []
for param in PARAMS:
    sub = agg[agg['parameter'] == param]

    line = alt.Chart(sub).mark_line(point=True).encode(
        x=alt.X('value:Q', title=param),
        y=alt.Y('mean:Q', title='Mean CGVQM Score', scale=alt.Scale(zero=False)),
        color=alt.Color('resolution:N', title='Base Resolution'),
        tooltip=['resolution', 'value', alt.Tooltip('mean:Q', format='.3f'), alt.Tooltip('ci95:Q', format='.3f')]
    )

    band = alt.Chart(sub).mark_errorband().encode(
        x='value:Q',
        y=alt.Y('mean:Q'),
        yError='ci95:Q',
        color='resolution:N'
    )

    charts.append((band + line).properties(title=f'Parameter: {param}', width=300, height=220))

alt.vconcat(*[alt.hconcat(*charts[:2]), alt.hconcat(*charts[2:])]).resolve_scale(color='shared')

alt.VConcatChart(...)